In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional, Sequence, Tuple
import warnings

import numpy as np
import pandas as pd


_SIDE_MAP = {
    "upper": "upper",
    "suction": "upper",
    "u": "upper",
    "top": "upper",
    "lower": "lower",
    "pressure": "lower",
    "l": "lower",
    "bottom": "lower",
}


def _standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    rename = {}
    for c in list(df.columns):
        low = c.lower().strip()
        if low in {"alpha", "aoa", "angle", "angle_of_attack"} and "Alpha" not in df.columns:
            rename[c] = "Alpha"
        elif low in {"cp", "c_p", "pressure_coefficient"} and "Cp" not in df.columns:
            rename[c] = "Cp"
        elif low in {"x/c", "x_c", "xc"} and "x" not in df.columns:
            rename[c] = "x"
        elif low in {"y/c", "y_c", "yc"} and "y" not in df.columns:
            rename[c] = "y"
        elif low in {"surface", "side", "surface_id"} and "side" not in df.columns:
            rename[c] = "side"
    return df.rename(columns=rename)


def add_surface_side(df: pd.DataFrame, side_column: str = "side") -> pd.DataFrame:
    """Add a canonical side column with values {'upper', 'lower'}.

    If a side/surface column already exists, it is mapped to canonical labels.
    Otherwise the fallback rule y >= 0 -> upper and y < 0 -> lower is used.
    For cambered airfoils, providing an explicit side/surface column is recommended.
    """
    df = df.copy()
    if side_column in df.columns:
        side_raw = df[side_column].astype(str).str.strip().str.lower()
        mapped = side_raw.map(_SIDE_MAP)
        if mapped.isna().any():
            bad = sorted(side_raw[mapped.isna()].unique().tolist())
            raise ValueError(f"Unrecognized surface labels in '{side_column}': {bad}")
        df["side"] = mapped
    else:
        if "y" not in df.columns:
            raise ValueError("A 'side' column or a 'y' column is required to identify the surface.")
        warnings.warn(
            "No explicit side/surface column found. Using y >= 0 as upper and y < 0 as lower. "
            "For cambered airfoils, provide a side column when possible.",
            RuntimeWarning,
            stacklevel=2,
        )
        df["side"] = np.where(df["y"].to_numpy(dtype=float) >= 0.0, "upper", "lower")
    return df


def load_labeled_dataframe(
    path: str | Path,
    require_cp: bool = True,
    side_column: str = "side",
) -> pd.DataFrame:
    """Load an Excel/CSV file with x, y, Alpha and optionally Cp.

    Required columns: x, y, Alpha. If require_cp=True, Cp is also required.
    Optional column: side/surface with labels upper/lower or suction/pressure.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}. Use .xlsx, .xls or .csv")

    df = _standardize_column_names(df)
    required = ["x", "y", "Alpha"] + (["Cp"] if require_cp else [])
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing columns {missing}. Found: {df.columns.tolist()}")

    keep = ["x", "y", "Alpha"] + (["Cp"] if "Cp" in df.columns else [])
    if side_column in df.columns:
        keep.append(side_column)
    elif "side" in df.columns and side_column != "side":
        keep.append("side")
        side_column = "side"

    df = df[keep].dropna().copy()
    for col in ["x", "y", "Alpha"]:
        df[col] = df[col].astype(float)
    if "Cp" in df.columns:
        df["Cp"] = df["Cp"].astype(float)

    df = add_surface_side(df, side_column=side_column if side_column in df.columns else "side")
    return df[[c for c in ["x", "y", "Alpha", "Cp", "side"] if c in df.columns]]


def split_by_alpha(
    df: pd.DataFrame,
    valid_alphas: Sequence[float],
    test_alphas: Sequence[float],
    atol: float = 1e-8,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create train/validation/test splits by angle of attack."""
    alpha = df["Alpha"].to_numpy(float)
    valid_mask = np.zeros(len(df), dtype=bool)
    test_mask = np.zeros(len(df), dtype=bool)
    for a in valid_alphas:
        valid_mask |= np.isclose(alpha, float(a), atol=atol)
    for a in test_alphas:
        test_mask |= np.isclose(alpha, float(a), atol=atol)
    if np.any(valid_mask & test_mask):
        raise ValueError("Validation and test alpha sets overlap.")
    train_mask = ~(valid_mask | test_mask)
    return df.loc[train_mask].copy(), df.loc[valid_mask].copy(), df.loc[test_mask].copy()
